# V16-11 | Copper: Paper-inspired reproduction vs Regime-adaptive forecasting

**Mục tiêu:** Chạy lại 7 phương pháp đã xây dựng ngày 14/11, đồng thời thử **Rolling Training**, **Recency Weighting** và **Residual (ΔCopper)** để đánh giá độ bền khi giá hàng hóa thay đổi chế độ. Các mô hình được huấn luyện lại trực tiếp trên CSV; **không yêu cầu file dự đoán cũ** như notebook 15/11.

**Ranh giới khoa học**

- Bài báo Chen et al. (2023) sử dụng **WTI, Gold, Silver** từ dữ liệu 1990–2009, công bố SA-LSTM 39/111 neurons, LR=5e-5, 100 epochs, batch 64, dropout 0.2. **Paper không xác định rõ lookback/horizon**, vì thế căn `X[t-21:t] → Copper(t)` là **giả định của notebook 14/11**, không phải điều tác giả xác nhận.
- `HORIZON=0` là **same-day nowcasting/estimation**: chỉ áp dụng khi biết giá đóng cửa của các biến ngoại sinh ngày `t` trước lúc ước lượng Copper(t). Không gọi kết quả này là dự báo ngày mai.
- Có thể đặt `HORIZON=1` để thử **dự báo đúng 1 ngày**: input chỉ đến `t-1` và target Copper(t). Khi đó phải đánh giá như thí nghiệm riêng.
- `paper3` = ba biến đúng tên bài báo; `legacy14` = bốn biến Aluminium, Tin, Silver, Gold của 14/11; `train_top4` = chọn top 4 Spearman **trên train riêng**, không xem trước test. **Không gộp các nhóm feature như một benchmark đồng nhất.**
- Hai cấu trúc `paper_sa_lstm` và `reoptimized_sa_lstm` của 14/11 không hoàn toàn giống nhau ở Dense/Dropout. Vì vậy khác biệt kết quả chưa thể quy hết cho SA. Giữ nguyên cách khởi tạo 14/11 theo yêu cầu, đồng thời có thêm thí nghiệm **matched architecture**.
- Nguồn dữ liệu hiện tại là Investing CSV 2016–2026, **không phải** dữ liệu gốc 1990–2009. Đây là *method reproduction on updated data*, không phải exact numerical replication.

**Chế độ chạy:** `QUICK_MODE` smoke test; `FULL_MODE` nghiên cứu thực nghiệm với SA; `PAPER_STYLE_MODE` tăng ngân sách SA (tốn GPU). Có checkpoint SA và lưu từng kết quả. Notebook không thể bảo đảm MSE thấp: đòi hỏi kiểm chứng trên Kaggle.

## A. Khái niệm của chiến lược V16

- **Distribution shift:** phân phối giá ở giai đoạn test lệch so với thời kỳ train (ví dụ nhiều giá test vượt max train). `MinMaxScaler` có thể trả giá trị **lớn hơn 1**; không nên clip vì làm mất tín hiệu.
- **Concept drift:** quan hệ dự đoán giữa các kim loại và giá đồng thay đổi. Tính Spearman từng cửa sổ để **chẩn đoán**, không dùng tương quan tương lai chọn đặc trưng.
- **Regime shift:** các trạng thái thị trường khác nhau (tăng trưởng/suy thoái/biến động mạnh) có thể dẫn đến hai hiện tượng trên.
- **Expanding:** train từ đầu lịch sử tới cutoff; **Rolling:** chỉ huấn luyện N phiên gần nhất tại mỗi cutoff (tại đây 756 ≈ 3 năm, 1260 ≈ 5 năm).
- **Recency weighting:** vẫn giữ lịch sử dài, nhưng tăng trọng số cho dữ liệu gần đây; `half_life=252` nghĩa là trọng số mẫu cách 252 phiên còn khoảng một nửa.
- **Residual learning:** thay vì học mức Copper trực tiếp, học `ΔCopper(t)=Copper(t)-Copper(t-1)` rồi cộng lại Copper(t-1). Đây là một **extension**, được phép dùng giá Copper hôm trước; không so sánh như cùng tập input với Paper3-exogenous-only.
- **Walk-forward:** liên tục dịch mốc huấn luyện/kiểm định và **fit lại** mô hình. Dùng ba holdout 90/92.5/95% mà không refit mỗi bước chưa phải walk-forward retraining hoàn chỉnh.
- **Nowcast vs forecast:** nếu dùng WTI(t) để dự đoán Copper(t), chỉ có ý nghĩa thực tế khi WTI(t) đã khả dụng; dự báo trước khi thị trường đóng cửa cần `HORIZON=1` hoặc biến đúng độ trễ.

## 1. Cấu hình và thư viện

In [ ]:
from pathlib import Path
import os, gc, json, time, math, hashlib, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
import tensorflow as tf

SEED = 42
MODE = 'FULL_MODE'  # QUICK_MODE | FULL_MODE | PAPER_STYLE_MODE
MODE_CFG = {
  'QUICK_MODE': {'epochs': 4, 'search_epochs': 3, 'sa_evals': 5, 'models_per_branch': 2, 'strategy_seeds': [42], 'earlystop': True},
  'FULL_MODE': {'epochs': 100, 'search_epochs': 20, 'sa_evals': 100, 'models_per_branch': 7, 'strategy_seeds': [17,42,91], 'earlystop': True},
  'PAPER_STYLE_MODE': {'epochs': 100, 'search_epochs': 50, 'sa_evals': 650, 'models_per_branch': 7, 'strategy_seeds': [17,42,91], 'earlystop': False},
}
CFG = MODE_CFG[MODE]
LOOKBACK = 22
HORIZON = 0            # 0: nowcast same-day, 1: next-day forecasting; DO NOT mix their scores
RATIOS = [0.90, 0.925, 0.95]
BATCH_SIZE = 64
FEATURE_MODES = ['paper3', 'legacy14']  # add 'train_top4' for separate ablation
RUN_ADAPTIVE = True
RUN_SA_GRU = True
RUN_WALK_FORWARD_CHECK = False  # True adds refitting on several pre-test origins
DATASET_CANDIDATES = [
    Path('/kaggle/input/datasets/lancelot71000/dataset-9-commodities-daily/dataset_9_commodities_daily.csv'),
    Path('/kaggle/input/dataset-9-commodities-daily/dataset_9_commodities_daily.csv'),
    Path('data/processed/dataset_9_commodities_daily.csv'),
]
OUT = Path('/kaggle/working/copper_v16_11') if Path('/kaggle/working').exists() else Path('copper_v16_11')
OUT.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass
print('mode:', MODE, '| horizon:', HORIZON, '| TF:', tf.__version__,'| GPU:',len(tf.config.list_physical_devices('GPU')))

## 2. Đọc dữ liệu một lần và kiểm tra chất lượng

In [ ]:
TARGET = 'Copper'
PAPER3 = ['WTI', 'Gold', 'Silver']
LEGACY14 = ['Aluminium', 'Tin', 'Silver', 'Gold']
CANDIDATES = ['Aluminium', 'Nickel', 'Tin', 'Iron_Ore_62', 'Silver', 'Gold', 'WTI', 'Brent']

DATA_PATH = next((p for p in DATASET_CANDIDATES if p.is_file()), None)
if DATA_PATH is None and Path('/kaggle/input').exists():
    matches = list(Path('/kaggle/input').rglob('dataset_9_commodities_daily.csv'))
    if len(matches) == 1: DATA_PATH = matches[0]
if DATA_PATH is None: raise FileNotFoundError('Không tìm thấy dataset_9_commodities_daily.csv; gắn đúng Kaggle dataset.')
raw = pd.read_csv(DATA_PATH)
if 'Date' not in raw.columns or TARGET not in raw.columns:
    raise KeyError('CSV phải có Date và Copper.')
raw['Date'] = pd.to_datetime(raw['Date'], errors='coerce')
for name in [TARGET]+CANDIDATES:
    if name in raw: raw[name] = pd.to_numeric(raw[name], errors='coerce')
if raw['Date'].isna().any(): raise ValueError('Date có dòng không hợp lệ.')
if raw['Date'].duplicated().any(): raise ValueError('Có ngày trùng lặp; xử lý nguồn dữ liệu trước.')
missing = sorted(set(PAPER3+LEGACY14)-set(raw.columns))
if missing: raise ValueError(f'Thiếu cột yêu cầu: {missing}')
available_candidates = [c for c in CANDIDATES if c in raw]
raw = raw.sort_values('Date').reset_index(drop=True)
quality = pd.DataFrame({'column':raw.columns,'missing':raw.isna().sum().values,'dtype':raw.dtypes.astype(str).values})
quality.to_csv(OUT/'data_quality.csv', index=False)
# Để so sánh các nhánh trên CÙNG ngày, loại những ngày thiếu bất kỳ biến ứng viên hiện có.
# Điều này là quy tắc làm sạch chủ động được ghi nhận, không phải chắc chắn như paper.
data = raw.dropna(subset=[TARGET]+available_candidates).reset_index(drop=True)
assert len(data)>1000 and data['Date'].is_monotonic_increasing
print('Dataset:',DATA_PATH, '| raw:',len(raw),'| clean:',len(data), '|',data.Date.iloc[0].date(),'->',data.Date.iloc[-1].date())
display(quality)

## 3. Quy tắc căn thời gian, feature selection và chống leakage

In [ ]:
def choose_features(frame, fit_start, val_start, feature_mode):
    train = frame.iloc[fit_start:val_start]
    if feature_mode == 'paper3':
        features = PAPER3.copy()
    elif feature_mode == 'legacy14':
        features = LEGACY14.copy()  # fixed a-priori; only for reproducing 14-11 setting
    elif feature_mode == 'train_top4':
        score = train[[TARGET]+available_candidates].corr(method='spearman')[TARGET].drop(TARGET)
        features = score.abs().sort_values(ascending=False).head(4).index.tolist()
    else: raise ValueError(feature_mode)
    corr = train[[TARGET]+available_candidates].corr(method='spearman')[TARGET].drop(TARGET)
    return features, corr

def prepare_context(frame, test_ratio, scheme='expanding', feature_mode='paper3', val_fraction=.10):
    n = len(frame)
    test_start = int(n*test_ratio)
    val_start = int(test_start*(1-val_fraction))
    assert LOOKBACK+HORIZON < val_start < test_start < n
    if scheme=='expanding' or scheme=='recency_weighted': fit_start=0
    elif scheme=='rolling_756': fit_start=max(0, val_start-756)
    elif scheme=='rolling_1260': fit_start=max(0, val_start-1260)
    else: raise ValueError(scheme)
    features, corr = choose_features(frame, fit_start, val_start, feature_mode)
    xsc = MinMaxScaler().fit(frame.iloc[fit_start:val_start][features])
    ysc = MinMaxScaler().fit(frame.iloc[fit_start:val_start][[TARGET]])
    x = xsc.transform(frame[features]).astype('float32')
    y = ysc.transform(frame[[TARGET]]).ravel().astype('float32')
    prev = np.r_[np.nan, frame[TARGET].to_numpy()[:-1]]
    prev_scaled = (prev - float(ysc.data_min_[0])) / float(ysc.data_range_[0])

    def windows(start,end, strictly_inside_train=False):
        idx = np.arange(start,end)
        idx = idx[(idx-HORIZON-LOOKBACK+1)>=0]
        if strictly_inside_train:
            idx = idx[(idx-HORIZON-LOOKBACK+1)>=fit_start]
        XX = np.stack([x[t-HORIZON-LOOKBACK+1:t-HORIZON+1] for t in idx]).astype('float32')
        YY = y[idx]
        return {'X':XX,'y':YY,'idx':idx,'prev':prev_scaled[idx].astype('float32'),
                'dates':frame.Date.iloc[idx].to_numpy()}
    train=windows(fit_start,val_start,True)
    val=windows(val_start,test_start)
    test=windows(test_start,n)
    assert max(train['idx'])<min(val['idx'])<min(test['idx'])
    assert np.max(train['idx']-HORIZON) < test_start
    assert xsc.n_samples_seen_ == val_start-fit_start
    return {'train':train,'val':val,'test':test,'x_scaler':xsc,'y_scaler':ysc,
            'features':features,'corr':corr,'fit_start':fit_start,'val_start':val_start,
            'test_start':test_start,'scheme':scheme,'ratio':test_ratio,'feature_mode':feature_mode}

# Independent smoke assertions for alignment, no test labels in input, scaler fit on train only.
ctx0=prepare_context(data,0.90)
assert ctx0['train']['X'].shape[1:] == (LOOKBACK,3)
assert len(set(ctx0['train']['idx']).intersection(ctx0['test']['idx']))==0
for ratio in RATIOS:
    c=prepare_context(data,ratio)
    print(f'{ratio:.3f}: train={len(c["train"]["idx"])} val={len(c["val"]["idx"])} test={len(c["test"]["idx"])} | {str(data.Date.iloc[c["test_start"]].date())}')

## 4. Drift diagnostics (chỉ dùng development period; không dùng test để chọn model)

In [ ]:
# Read-only drift diagnostics; may inspect held-out data AFTER model configs are frozen.
split0=prepare_context(data,0.90)
corr_rows=[]
dev_data = data.iloc[:split0['test_start']].copy()  # restrict diagnostics before held-out test
for start in range(0, len(dev_data)-252+1, 63):
    part = dev_data.iloc[start:start+252]
    for feat in PAPER3:
        corr_rows.append({'window_end':part.Date.iloc[-1], 'feature':feat,
                          'spearman_252':float(part[[TARGET,feat]].corr(method='spearman').iloc[0,1])})
correlation_over_time=pd.DataFrame(corr_rows)
correlation_over_time.to_csv(OUT/'spearman_rolling_diagnostic.csv',index=False)
fig,ax=plt.subplots(figsize=(10,4))
for col,part in correlation_over_time.groupby('feature'):
    ax.plot(part.window_end,part.spearman_252,label=col)
ax.axhline(0,color='gray',linewidth=.6);ax.set(title='252-session rolling Spearman (diagnostic only)',ylabel='Spearman')
ax.legend();fig.tight_layout();fig.savefig(OUT/'rolling_spearman.png',dpi=160);plt.show()
train_part=data.iloc[:split0['val_start']];test_part=data.iloc[split0['val_start']:split0['test_start']] # validation diagnostic only
shift_report=[]
for col in [TARGET]+PAPER3:
    v=train_part[col];z=test_part[col]
    shift_report.append({'column':col,'train_min':v.min(),'train_max':v.max(),'validation_min':z.min(),'validation_max':z.max(),
                         'validation_outside_training_range_fraction':float(((z<v.min())|(z>v.max())).mean())})
shift_df=pd.DataFrame(shift_report)
shift_df.to_csv(OUT/'validation_distribution_shift_diagnostic.csv',index=False)
display(shift_df)

## 5. Bảy mô hình: giữ nguyên kiến trúc từ 14/11

Các định nghĩa dưới đây được giữ theo phiên bản 14/11. Riêng paper SA-LSTM chứa giả định Dense 32/ReLU và Linear output (bài báo không công bố toàn bộ activation). Sự khác nhau về Dense/Dropout giữa paper và reoptimized sẽ được kiểm soát trong ablation riêng.

In [ ]:
def build_lstm(input_shape, hp=None):
    model = tf.keras.Sequential(name='LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(64, return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(32))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(16, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

In [ ]:
PAPER_SA_LSTM_HP = {'units_1': 39, 'units_2': 111, 'learning_rate': 5e-5}

def build_paper_sa_lstm(input_shape, hp=None):
    # Hai lớp LSTM và bộ 39/111, LR=5e-5 được paper công bố.
    # Dense ReLU, output linear và Adam là các giả định triển khai đã được ghi nhận.
    model = tf.keras.Sequential(name='Paper_SA_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(39, return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(111))
    model.add(tf.keras.layers.Dense(32, activation='relu'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-5), loss='mse')
    return model

In [ ]:
def build_reoptimized_sa_lstm(input_shape, hp):
    model = tf.keras.Sequential(name='Reoptimized_SA_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(int(hp['units_1']), return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(int(hp['units_2'])))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(16, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    optimizer = tf.keras.optimizers.Adam(float(hp['learning_rate']))
    model.compile(optimizer=optimizer, loss='mse')
    return model

In [ ]:
def build_gru(input_shape, hp=None):
    model = tf.keras.Sequential(name='GRU')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.GRU(64, return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.GRU(32))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(16, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

In [ ]:
def build_cnn1d_lstm(input_shape, hp=None):
    model = tf.keras.Sequential(name='CNN1D_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.Conv1D(filters=32, kernel_size=3, padding='causal', activation='relu'))
    model.add(tf.keras.layers.MaxPooling1D(pool_size=2))
    model.add(tf.keras.layers.LSTM(48, return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(24))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(16, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

In [ ]:
def build_sa_gru(input_shape, hp):
    model = tf.keras.Sequential(name='SA_GRU')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.GRU(int(hp['units_1']), return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.GRU(int(hp['units_2'])))
    model.add(tf.keras.layers.Dense(16, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    optimizer = tf.keras.optimizers.Adam(float(hp['learning_rate']))
    model.compile(optimizer=optimizer, loss='mse')
    return model

### Kiểm soát nhiễu kiến trúc: matched SA-LSTM

Để đánh giá **tác động riêng của hyperparameters**, hai cấu hình SA phải dùng **cùng cấu trúc Dense/Dropout**. Đây là nhánh kiểm soát, không thay thế bảy kiến trúc gốc.

*Lưu ý:* hyperparameters mới được SA tìm từ kiến trúc gốc; chuyển sang kiến trúc matched là kiểm tra độ nhạy, chưa phải SA tối ưu lại ngay trên kiến trúc matched.

In [ ]:
def build_matched_sa_lstm(input_shape,hp):
    model=tf.keras.Sequential(name='Matched_SA_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(int(hp['units_1']),return_sequences=True))
    model.add(tf.keras.layers.Dropout(.20))
    model.add(tf.keras.layers.LSTM(int(hp['units_2'])))
    model.add(tf.keras.layers.Dense(32,activation='relu'))
    model.add(tf.keras.layers.Dropout(.20))
    model.add(tf.keras.layers.Dense(1,activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(float(hp['learning_rate'])),loss='mse')
    return model

def build_cnn_gru(input_shape,hp=None):
    # CNN-GRU strategy from 15-11; simple fixed comparison, no expensive grid search here.
    model=tf.keras.Sequential(name='CNN_GRU')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.Conv1D(32,3,padding='causal',activation='relu'))
    model.add(tf.keras.layers.MaxPooling1D(2))
    model.add(tf.keras.layers.GRU(48,return_sequences=True))
    model.add(tf.keras.layers.GRU(24))
    model.add(tf.keras.layers.Dense(16,activation='relu'))
    model.add(tf.keras.layers.Dropout(.2))
    model.add(tf.keras.layers.Dense(1,activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='mse')
    return model

## 6. Huấn luyện, đánh giá, huấn luyện sai phân (residual)

In [ ]:
def reset_keras(seed=SEED):
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(int(seed))

def weights_for_context(ctx):
    n=len(ctx['train']['idx']); age=np.arange(n-1,-1,-1,dtype=float)
    w=np.exp(-np.log(2)*age/252)
    return (w/np.mean(w)).astype('float32')

def fit_predict(builder,ctx,hp=None,seed=SEED,epochs=None,scheme=None,residual=False, search=False, skip_test=False):
    reset_keras(seed)
    tr,va = ctx['train'],ctx['val']
    te = None if skip_test else ctx['test']
    model = builder(tr['X'].shape[1:],hp)
    callbacks=[]
    if CFG['earlystop']:
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=7 if not search else 4,restore_best_weights=True)]
    ytr=tr['y']-tr['prev'] if residual else tr['y']
    yva=va['y']-va['prev'] if residual else va['y']
    weights=weights_for_context(ctx) if scheme=='recency_weighted' else None
    begin=time.perf_counter()
    hist=model.fit(tr['X'],ytr,validation_data=(va['X'],yva),epochs=epochs or CFG['epochs'],
                   batch_size=BATCH_SIZE,shuffle=False,callbacks=callbacks,verbose=0,sample_weight=weights)
    seconds=time.perf_counter()-begin
    pv=model.predict(va['X'],verbose=0).ravel()
    pt=model.predict(te['X'],verbose=0).ravel() if te is not None else None
    if residual:
        pv=pv+va['prev']
        if te is not None: pt=pt+te['prev']
    val_mse=float(mean_squared_error(va['y'],pv))
    ret={'val_pred':pv,'test_pred':pt,'val_mse':val_mse,'seconds':seconds,'epochs_run':len(hist.history['loss']),
         'loss_history':hist.history}
    del model;reset_keras(seed)
    return ret

def ridge_residual_prediction(ctx):
    tr,va,te=ctx['train'],ctx['val'],ctx['test']
    model=Ridge(alpha=1.)
    model.fit(tr['X'].reshape(len(tr['X']),-1),tr['y']-tr['prev'])
    pv=va['prev']+model.predict(va['X'].reshape(len(va['X']),-1))
    pt=te['prev']+model.predict(te['X'].reshape(len(te['X']),-1))
    return {'val_pred':pv,'test_pred':pt,'val_mse':float(mean_squared_error(va['y'],pv)),
            'seconds':0.,'epochs_run':0,'loss_history':None}

def evaluate_predictions(ctx,pred,model,seed,branch,scheme):
    true=ctx['test']['y'];ysc=ctx['y_scaler']
    true_raw=ysc.inverse_transform(true.reshape(-1,1)).ravel()
    pred_raw=ysc.inverse_transform(np.asarray(pred['test_pred']).reshape(-1,1)).ravel()
    persistence_scaled=ctx['test']['prev']
    naive_mae=float(mean_absolute_error(true,persistence_scaled))
    row={'branch':branch,'feature_mode':ctx['feature_mode'],'scheme':scheme,'ratio':ctx['ratio'],
         'horizon':HORIZON,'model':model,'seed':seed,'features':','.join(ctx['features']),
         'n_test':len(true),'train_start':str(data.Date.iloc[ctx['fit_start']].date()),
         'test_start':str(data.Date.iloc[ctx['test_start']].date()),
         'val_mse_scaled':pred['val_mse'],'MSE_scaled':float(mean_squared_error(true,pred['test_pred'])),
         'RMSE_raw':float(mean_squared_error(true_raw,pred_raw)**.5),
         'MAE_raw':float(mean_absolute_error(true_raw,pred_raw)),
         'R2_raw':float(r2_score(true_raw,pred_raw)),
         'Relative_MAE_vs_test_persistence':float(mean_absolute_error(true,pred['test_pred'])/naive_mae) if naive_mae>0 else np.nan,
         'MASE':float(mean_absolute_error(true_raw,pred_raw)/np.mean(np.abs(np.diff(data[TARGET].iloc[ctx['fit_start']:ctx['val_start']].to_numpy())))),
         'epochs_run':pred['epochs_run'],'train_seconds':pred['seconds']}
    preds=pd.DataFrame({'Date':ctx['test']['dates'],'ratio':ctx['ratio'],'branch':branch,'scheme':scheme,'feature_mode':ctx['feature_mode'],
                         'model':model,'seed':seed,'actual_raw':true_raw,'pred_raw':pred_raw})
    return row,preds

RESULT_ROWS=[]; PREDICTIONS=[]
def record_result(ctx, pred, model, seed, branch, scheme):
    row,frame=evaluate_predictions(ctx,pred,model,seed,branch,scheme)
    RESULT_ROWS.append(row);PREDICTIONS.append(frame)
    pd.DataFrame(RESULT_ROWS).to_csv(OUT/'results_v16.csv',index=False)
    pd.concat(PREDICTIONS,ignore_index=True).to_csv(OUT/'predictions_v16.csv',index=False)
    print(f'[{branch}/{scheme}/{ctx["ratio"]}] {model}, seed={seed}: raw_RMSE={row["RMSE_raw"]:.5f}, MSE_scaled={row["MSE_scaled"]:.6f}')

## 7. Simulated Annealing độc lập trên train/validation

LSTM và GRU được tìm hyperparameters trên **split 90/10, phần train/validation trước khi test bắt đầu**. Không dùng test tìm HP. Công thức chấp nhận Metropolis dùng `exp(-ΔE/T)` (dấu âm), phù hợp với việc tối thiểu hóa lỗi. Giữ proposal dạng sigmoid latent của phiên bản 14/11; bounds và cách chia ngân sách là giả định.

In [ ]:
def fit_validation_only(builder,ctx,hp,epochs):
    r=fit_predict(builder,ctx,hp=hp,epochs=epochs,search=True,skip_test=True)
    return r['val_mse'],r['seconds'],r['epochs_run']

In [ ]:
PAPER_SA = {
    'initial_temperature': 1.0, 'final_temperature': 0.1, 'cooling': 0.95,
    'proposals_per_temperature': 15,  # 650 scenarios / khoảng 45 mức nhiệt
    'proposal_width': 2.0, 'units_min': 8, 'units_max': 128,
    'lr_min': 1e-6, 'lr_max': 0.005,
    'initial': {'units_1': 15, 'units_2': 30, 'learning_rate': 5e-4},
}

def _sigmoid(z):
    return 1.0 / (1.0 + math.exp(-float(np.clip(z, -30, 30))))

def _logit(p):
    p = float(np.clip(p, 1e-6, 1 - 1e-6))
    return math.log(p / (1.0 - p))

def paper_proposal(current, temperature, rng):
    # Bám Eq. 7–8: chuyển tham số sang latent coordinate, perturb theo T,
    # rồi sigmoid trở lại miền hữu hạn. Bounds/proposal_width là giả định cần thiết.
    candidate = {}
    for key in ('units_1', 'units_2', 'learning_rate'):
        if key == 'learning_rate':
            lo, hi, value = math.log(PAPER_SA['lr_min']), math.log(PAPER_SA['lr_max']), math.log(current[key])
        else:
            lo, hi, value = PAPER_SA['units_min'], PAPER_SA['units_max'], current[key]
        latent = _logit((value - lo) / (hi - lo))
        moved = latent + temperature * PAPER_SA['proposal_width'] * rng.uniform(-1.0, 1.0)
        bounded = lo + (hi - lo) * _sigmoid(moved)
        candidate[key] = float(math.exp(bounded)) if key == 'learning_rate' else int(np.clip(round(bounded), lo, hi))
    return candidate

def metropolis(delta, temperature, rng):
    # Paper in exp(+delta/T), nhưng dấu đó cho xác suất > 1 với nghiệm xấu.
    # Dùng dấu âm chuẩn của Metropolis; đây là hiệu chỉnh lỗi công thức được công bố.
    probability = 1.0 if delta <= 0 else math.exp(-delta / max(temperature, 1e-12))
    return bool(rng.random() < probability), probability

def _save_sa_checkpoint(path, state):
    path.write_text(json.dumps(state, indent=2, default=str), encoding='utf-8')

def simulated_annealing_search(search_name, builder, ctx):
    checkpoint = OUT / f'{search_name}_paper_sa_checkpoint.json'
    history_path = OUT / f'{search_name}_paper_sa_history.csv'
    rng = np.random.default_rng(SEED)
    if checkpoint.exists():
        state = json.loads(checkpoint.read_text(encoding='utf-8'))
        rng.bit_generator.state = state['rng_state']
        print(f'Resume {search_name} từ evaluation {state["evaluations"]}')
    else:
        current = dict(PAPER_SA['initial'])
        loss, seconds, epochs_run = fit_validation_only(builder, ctx, current, CFG['search_epochs'])
        state = {
            'current': current, 'current_loss': loss, 'best': dict(current), 'best_loss': loss,
            'evaluations': 1, 'history': [{'evaluation': 1, 'temperature': 1.0,
                'val_mse_scaled': loss, 'best_mse_scaled': loss, 'accepted': True,
                'acceptance_probability': 1.0, **current, 'epochs_run': epochs_run,
                'train_seconds': seconds}],
        }
    while state['evaluations'] < CFG['sa_evals']:
        proposal_number = state['evaluations'] - 1
        temperature = PAPER_SA['initial_temperature'] * PAPER_SA['cooling'] ** (
            proposal_number // PAPER_SA['proposals_per_temperature'])
        if temperature < PAPER_SA['final_temperature']:
            break
        candidate = paper_proposal(state['current'], temperature, rng)
        loss, seconds, epochs_run = fit_validation_only(builder, ctx, candidate, CFG['search_epochs'])
        accepted, probability = metropolis(loss - state['current_loss'], temperature, rng)
        if accepted:
            state['current'], state['current_loss'] = candidate, loss
        if loss < state['best_loss']:
            state['best'], state['best_loss'] = dict(candidate), loss
        state['evaluations'] += 1
        state['history'].append({'evaluation': state['evaluations'], 'temperature': temperature,
            'val_mse_scaled': loss, 'best_mse_scaled': state['best_loss'], 'accepted': accepted,
            'acceptance_probability': probability, **candidate, 'epochs_run': epochs_run,
            'train_seconds': seconds})
        state['rng_state'] = rng.bit_generator.state
        _save_sa_checkpoint(checkpoint, state)
        pd.DataFrame(state['history']).to_csv(history_path, index=False)
        print(f'{search_name} {state["evaluations"]}/{cfg["sa_evaluations"]}: T={temperature:.4f}, val={loss:.6f}, best={state["best_loss"]:.6f}')
    result = {'model': search_name, 'best_hp': state['best'],
              'best_val_mse_scaled': state['best_loss'], 'evaluations': state['evaluations']}
    (OUT / f'{search_name}_best_hp.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    return state['best'], pd.DataFrame(state['history'])

## 8. Chạy SA mới và lưu checkpoint

Sau khi chạy 1 lần, checkpoint được nạp lại khi chạy cell. Nếu đổi feature, horizon, lookback hoặc mode, hãy dùng **thư mục output mới** hoặc xóa checkpoint cũ; cell này tự thêm hash cấu hình vào tên để không trộn ngữ cảnh.

In [ ]:
sa_cache={}
for feature_mode in FEATURE_MODES:
    primary=prepare_context(data,0.90,'expanding',feature_mode)
    tag=f'{feature_mode}_h{HORIZON}_w{LOOKBACK}_{MODE}'
    hp,history=simulated_annealing_search('sa_lstm_'+tag,build_reoptimized_sa_lstm,primary)
    sa_cache[feature_mode]={'lstm':hp}
    if RUN_SA_GRU:
        ghp,ghist=simulated_annealing_search('sa_gru_'+tag,build_sa_gru,primary)
        sa_cache[feature_mode]['gru']=ghp
(OUT/'sa_best_hyperparameters.json').write_text(json.dumps(sa_cache,indent=2),encoding='utf-8')
display(sa_cache)

## 9. Nhánh A: retrain 7 mô hình theo chiến lược phiên bản 14/11

Mọi mô hình trong mỗi branch dùng **cùng input features, ngày train/val/test và horizon**. Cấu hình Paper SA là hyperparameters công bố, không phải chạy lại SA. Kết quả trong `paper3` là đối chiếu *paper-inspired*; `legacy14` là cách chọn biến ngày 14/11.

In [ ]:
PAPER_HP={'units_1':39,'units_2':111,'learning_rate':5e-5}
for feature_mode in FEATURE_MODES:
  for ratio in RATIOS:
    ctx=prepare_context(data,ratio,'expanding',feature_mode)
    naive={'test_pred':ctx['test']['prev'],'val_mse':float(mean_squared_error(ctx['val']['y'],ctx['val']['prev'])),
           'seconds':0.,'epochs_run':0}
    record_result(ctx,naive,'naive_persistence',SEED,'A_original_models','expanding')
    models=[('lstm',build_lstm,None),('paper_sa_lstm',build_paper_sa_lstm,None),
            ('reoptimized_sa_lstm',build_reoptimized_sa_lstm,sa_cache[feature_mode]['lstm']),
            ('gru',build_gru,None),('cnn1d_lstm',build_cnn1d_lstm,None)]
    if RUN_SA_GRU:models.append(('sa_gru',build_sa_gru,sa_cache[feature_mode]['gru']))
    if MODE=='QUICK_MODE':models=models[:CFG['models_per_branch']]
    for name,builder,hp in models:
        p=fit_predict(builder,ctx,hp=hp)
        record_result(ctx,p,name,SEED,'A_original_models','expanding')
    # Matched SA-LSTM ablation: isolate HP differences, same Dense/Dropout backbone.
    if MODE!='QUICK_MODE':
        for name,hp in [('matched_paper',PAPER_HP),('matched_reoptimized',sa_cache[feature_mode]['lstm'])]:
            p=fit_predict(build_matched_sa_lstm,ctx,hp=hp)
            record_result(ctx,p,name,SEED,'A_matched_architecture','expanding')

## 10. Nhánh B: chiến lược thích nghi dữ liệu bất ổn định

**Giữ nguyên kiến trúc 14/11** (`paper_sa_lstm`, `gru`) và thử 3 chiến lược học khác nhau: Expanding, Rolling 756, Recency-weighted. Thêm Residual-GRU/Ridge (extension) dùng giá đồng ngày trước. Chỉ dùng `paper3` để kiểm soát số biến; không tự tuyên bố extension thuộc bài báo.

**Quan trọng:** Đây là các phương án **được đánh giá**, không bảo đảm tốt hơn. Chạy ba random seeds giúp thấy độ ổn định; so sánh MASE (chuẩn với mẫu số train), test-relative-MAE/Persistence và giá gốc, không chỉ MSE MinMax.

In [ ]:
if RUN_ADAPTIVE:
  for ratio in RATIOS:
    for scheme in ['expanding','rolling_756','recency_weighted']:
      ctx=prepare_context(data,ratio,scheme,'paper3')
      for seed in CFG['strategy_seeds']:
        for name,builder in [('paper_sa_lstm',build_paper_sa_lstm),('gru',build_gru),('cnn_gru',build_cnn_gru)]:
            p=fit_predict(builder,ctx,seed=seed,scheme=scheme)
            record_result(ctx,p,name,seed,'B_adaptation',scheme)
        # Residual-learning GRU: trains on Δscaled Copper(t), adds actual Copper(t−1) at inference.
        p=fit_predict(build_gru,ctx,seed=seed,scheme=scheme,residual=True)
        record_result(ctx,p,'residual_gru',seed,'B_residual_extension',scheme)
      ridge=ridge_residual_prediction(ctx)
      record_result(ctx,ridge,'ridge_delta',None,'B_residual_extension',scheme)

## 11. Walk-forward retraining thực sự — tuỳ chọn, chỉ trong development period

Không dùng test 10% cuối cho lựa chọn chiến lược. Mỗi origin trước mốc test cuối là một bài toán **fit lại hoàn toàn** và đánh giá trên block tiếp theo. Số bước thấp để không quá tốn GPU.

In [ ]:
def walk_forward_pretest(model_builder=build_gru,feature_mode='paper3',scheme='expanding',n_folds=3,block=80):
    dev_end=int(len(data)*.90)  # reserve last 10% untouched by walk-forward model selection
    origs=np.linspace(int(dev_end*.65),dev_end-block,n_folds,dtype=int)
    records=[]
    for origin in origs:
        # Temporary development frame ends at origin+block; its test begins exactly at origin
        sub=data.iloc[:origin+block].reset_index(drop=True)
        ratio=origin/len(sub)
        c=prepare_context(sub,ratio,scheme,feature_mode,val_fraction=.10)
        p=fit_predict(model_builder,c,seed=SEED,epochs=CFG['epochs'],scheme=scheme)
        truth=c['y_scaler'].inverse_transform(c['test']['y'].reshape(-1,1)).ravel()
        predict=c['y_scaler'].inverse_transform(p['test_pred'].reshape(-1,1)).ravel()
        records.append({'origin':str(sub.Date.iloc[origin].date()),'scheme':scheme,
                        'RMSE_raw':float(mean_squared_error(truth,predict)**.5),'n_test':len(truth)})
    return pd.DataFrame(records)

if RUN_WALK_FORWARD_CHECK:
    wf=pd.concat([walk_forward_pretest(scheme=s) for s in ['expanding','rolling_756']],ignore_index=True)
    wf.to_csv(OUT/'pretest_walkforward_results.csv',index=False)
    display(wf)

## 12. Tổng hợp kết quả và biểu đồ

Không chọn chiến lược bằng cách nhìn MSE test: so sánh validation hoặc walk-forward pre-test để quyết định mô hình trước khi kết luận về holdout cuối. Các mốc 90%, 92.5%, 95% có **test chồng lấn**, không phải 3 mẫu độc lập cho kiểm định thống kê.

In [ ]:
results=pd.DataFrame(RESULT_ROWS)
predictions=pd.concat(PREDICTIONS,ignore_index=True)
results.to_csv(OUT/'results_v16.csv',index=False)
predictions.to_csv(OUT/'predictions_v16.csv',index=False)
pivot=results.groupby(['branch','feature_mode','scheme','model','ratio'],dropna=False)[['MSE_scaled','RMSE_raw','MAE_raw','MASE','Relative_MAE_vs_test_persistence']].mean().reset_index()
pivot.to_csv(OUT/'summary_v16.csv',index=False)
print('Primary holdout (90/10), top 20 lowest raw RMSE:')
display(pivot[np.isclose(pivot.ratio,.90)].sort_values('RMSE_raw').head(20))
fig,axes=plt.subplots(1,2,figsize=(14,5))
primary=pivot[(np.isclose(pivot.ratio,.90)) & (pivot.feature_mode=='paper3')]
for scheme,part in primary.groupby('scheme'):
    part=part.sort_values('RMSE_raw')
    axes[0].plot(part.model,part.RMSE_raw,'o-',label=scheme)
axes[0].set(title='RMSE_raw (90/10) - paper3',ylabel='RMSE in copper price units')
axes[0].tick_params(axis='x',rotation=75);axes[0].legend(fontsize=8)
for scheme,part in primary.groupby('scheme'):
    part=part.sort_values('MASE','Relative_MAE_vs_test_persistence')
    axes[1].plot(part.model,part.Relative_MAE_vs_test_persistence,'o-',label=scheme)
axes[1].axhline(1,color='gray',linestyle='--',label='persistence baseline')
axes[1].set(title='Test MAE / Persistence test MAE (lower is better)',ylabel='relative test MAE')
axes[1].tick_params(axis='x',rotation=75);axes[1].legend(fontsize=8)
fig.tight_layout();fig.savefig(OUT/'strategy_comparison_v16.png',dpi=170,bbox_inches='tight');plt.show()

# Example predictions for a controlled comparison
show=predictions[(np.isclose(predictions.ratio,.90)) & (predictions.feature_mode=='paper3') & (predictions.branch=='A_original_models') &
                 (predictions.scheme=='expanding') & (predictions.model.isin(['naive_persistence','lstm','paper_sa_lstm','gru']))].copy()
if len(show):
    grp=show[show.model.isin(['naive_persistence','lstm','paper_sa_lstm','gru'])]
    fig,ax=plt.subplots(figsize=(14,5))
    actual=grp[grp.model=='naive_persistence']
    ax.plot(actual.Date,actual.actual_raw,label='Copper actual',color='black',linewidth=1.5)
    for name,part in grp.groupby('model'):
        ax.plot(part.Date,part.pred_raw,label=name,linewidth=1,alpha=.85)
    ax.set(title='V16 paper3 | 90/10 split | same test dates',ylabel='Copper price')
    ax.legend();fig.tight_layout();fig.savefig(OUT/'actual_vs_predicted_v16.png',dpi=160);plt.show()
# POST-HOC distribution shift (inspect test only after all model results have been produced)
posthoc_train=data.iloc[:split0['val_start']]
posthoc_test=data.iloc[split0['test_start']:]
posthoc=[]
for col in [TARGET]+PAPER3:
    a=posthoc_train[col];b=posthoc_test[col]
    posthoc.append({'feature':col,'train_min':a.min(),'train_max':a.max(),'test_min':b.min(),'test_max':b.max(),
                    'fraction_outside_train_range':float(((b<a.min())|(b>a.max())).mean())})
posthoc=pd.DataFrame(posthoc)
posthoc.to_csv(OUT/'posthoc_test_shift_diagnostics.csv',index=False)
display(posthoc)


## 13. Kết luận cần dựa trên bằng chứng

Trước khi viết báo cáo: (1) nếu Relative_MAE_vs_test_persistence > 1 thì thua baseline Copper hôm qua trên test; MASE dùng mẫu số sai số naive trong train; (2) nếu rolling giảm RMSE qua validation/walk-forward pre-test, có thể đề xuất dữ liệu gần đây hữu ích hơn dữ liệu cũ; (3) nếu residual-GRU tốt hơn GRU nhưng dùng thêm Copper(t−1), phải ghi đó là **thêm thông tin đầu vào**; (4) không dùng số MSE gốc bài báo trên 1990–2009 như kết quả cạnh tranh trực tiếp với dữ liệu 2016–2026.

**Giới hạn:** Không có cơ sở xác nhận paper gốc căn biến cùng ngày, lookback=22, hay cách họ định nghĩa Dense output. Bảy model ở nhánh A là cách triển khai 14/11; 'paper3' chỉ làm cho lựa chọn biến gần paper hơn. Không đánh đồng paper-inspired với exact reproduction.

**Các đầu ra:** `results_v16.csv`, `predictions_v16.csv`, `summary_v16.csv`, `sa_best_hyperparameters.json`, lịch sử SA, báo cáo drift, biểu đồ. Mọi kết quả phải được sinh từ Kaggle Run All, không điền số giả.